# Notebook 5 — Promotion EDA

**Goal:** Measure promotion lift, detect the post-promotion demand dip, and begin thinking about cannibalization.

This is a preview of Project 5 (The Promotion Trap). The key lesson:

> A promotion that looks like a success in week 1 may destroy margin in weeks 2–4 through the post-promotion dip, forward buying, and inventory build.

**Core business questions:**
1. Do promotions actually drive incremental demand, or just pull forward future purchases?
2. Does the dip after a promotion erase the gains?
3. Which promotion types have the highest lift?
4. Do promotions cause stockouts (demand exceeds forecast)?

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sqlalchemy import create_engine

DB_URL = os.getenv('SUPPLY_CHAIN_DB_URL', '')
engine = create_engine(DB_URL)
SCHEMA = 'supply_chain'

def q(sql): return pd.read_sql(sql, engine)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Promotion coverage — how many products are promoted each week?

In [ ]:
promo_coverage = q(f"""
SELECT
    promotion_type,
    COUNT(*) AS total_promotions,
    COUNT(DISTINCT product_id) AS distinct_products,
    ROUND(AVG(discount_pct * 100), 1) AS avg_discount_pct,
    ROUND(AVG(end_date - start_date + 1), 1) AS avg_duration_days
FROM {SCHEMA}.promotions
GROUP BY promotion_type
ORDER BY total_promotions DESC
""")
print(promo_coverage.to_string(index=False))

## 2. Measured promotion lift using the analytical view

The `v_promotion_lift` view compares promoted sales vs. the 4-week pre-promotion baseline.

A lift factor of 1.5 means the product sold 50% more units during the promotion than the baseline.

In [ ]:
lift = q(f"""
SELECT promotion_type, category,
       COUNT(*) AS n,
       ROUND(AVG(lift_factor), 3) AS avg_lift,
       ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY lift_factor), 3) AS p25_lift,
       ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY lift_factor), 3) AS p75_lift
FROM {SCHEMA}.v_promotion_lift
WHERE lift_factor IS NOT NULL AND lift_factor > 0
GROUP BY promotion_type, category
ORDER BY avg_lift DESC
LIMIT 20
""")
print(lift.to_string(index=False))

In [ ]:
# Lift by promotion type — box plot
lift_raw = q(f"""
SELECT promotion_type, lift_factor
FROM {SCHEMA}.v_promotion_lift
WHERE lift_factor IS NOT NULL AND lift_factor BETWEEN 0.5 AND 5.0
""")

promo_types = lift_raw['promotion_type'].unique().tolist()
groups = [lift_raw[lift_raw['promotion_type'] == t]['lift_factor'].values for t in promo_types]

fig, ax = plt.subplots()
ax.boxplot(groups, labels=promo_types, patch_artist=True)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, label='No lift')
ax.set_title('Promotion Lift Factor Distribution by Type')
ax.set_ylabel('Lift Factor (1.0 = no effect)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Post-promotion demand dip

This is the "trap" in Project 5. After a promotion ends, demand often drops below the baseline as customers have stocked up (forward buying) or the product has lost its "deal" signal.

We look at a single product's daily sales window: 2 weeks pre-promo, during promo, 2 weeks post-promo.

In [ ]:
# Find a product with a clear short promotion (7–14 days)
sample_promo = q(f"""
SELECT pr.promotion_id, pr.product_id, pr.promotion_name, pr.promotion_type,
       pr.start_date, pr.end_date, pr.discount_pct,
       p.product_name, p.category
FROM {SCHEMA}.promotions pr
JOIN {SCHEMA}.products p ON p.product_id = pr.product_id
WHERE (pr.end_date - pr.start_date) BETWEEN 6 AND 14
  AND pr.start_date BETWEEN '2023-03-01' AND '2023-09-30'
LIMIT 1
""")
print(sample_promo[['promotion_name','product_name','category','start_date','end_date','discount_pct']])

pid       = int(sample_promo.iloc[0]['product_id'])
promo_start = sample_promo.iloc[0]['start_date']
promo_end   = sample_promo.iloc[0]['end_date']

In [ ]:
# Daily sales for that product, 28 days pre to 28 days post promotion
window = q(f"""
SELECT
    st.date_id,
    SUM(st.quantity_sold) AS total_units,
    BOOL_OR(st.promotion_id IS NOT NULL) AS is_promo
FROM {SCHEMA}.sales_transactions st
WHERE st.product_id = {pid}
  AND st.date_id BETWEEN '{promo_start}'::date - 28 AND '{promo_end}'::date + 28
GROUP BY st.date_id
ORDER BY st.date_id
""")
window['date_id'] = pd.to_datetime(window['date_id'])
promo_start_dt = pd.to_datetime(promo_start)
promo_end_dt   = pd.to_datetime(promo_end)

fig, ax = plt.subplots()
colors = ['#F44336' if r['is_promo'] else '#2196F3' for _, r in window.iterrows()]
ax.bar(window['date_id'], window['total_units'], color=colors, width=0.8)
ax.axvspan(promo_start_dt, promo_end_dt, alpha=0.1, color='red', label='Promotion period')
ax.axvline(promo_end_dt, color='red', linestyle='--', linewidth=1)

pre_avg  = window[window['date_id'] < promo_start_dt]['total_units'].mean()
post_avg = window[window['date_id'] > promo_end_dt]['total_units'].mean()
ax.axhline(pre_avg,  color='blue',  linestyle=':', linewidth=1.5, label=f'Pre avg: {pre_avg:.1f}')
ax.axhline(post_avg, color='green', linestyle=':', linewidth=1.5, label=f'Post avg: {post_avg:.1f}')

ax.set_title(f'Daily Sales Around Promotion\n{sample_promo.iloc[0]["product_name"]} ({sample_promo.iloc[0]["category"]})')
ax.set_xlabel('Date')
ax.set_ylabel('Units Sold')
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(f'Pre-promo average:  {pre_avg:.2f} units/day')
print(f'Post-promo average: {post_avg:.2f} units/day')
print(f'Post-promo dip:     {(post_avg - pre_avg) / pre_avg * 100:+.1f}%')

## 4. Do promotions cause stockouts?

If promotions drive unexpected demand that inventory can't absorb, we'll see elevated stockout rates on promoted products.

In [ ]:
promo_stockout = q(f"""
SELECT
    CASE WHEN st.promotion_id IS NOT NULL THEN 'Promoted' ELSE 'Not Promoted' END AS promo_status,
    COUNT(*) AS transactions,
    SUM(CASE WHEN st.is_stockout_day THEN 1 ELSE 0 END) AS stockout_trans,
    ROUND(100.0 * SUM(CASE WHEN st.is_stockout_day THEN 1 ELSE 0 END) / COUNT(*), 3) AS stockout_rate_pct
FROM {SCHEMA}.sales_transactions st
GROUP BY promo_status
""")
print(promo_stockout.to_string(index=False))

## 5. Promotion discount depth vs. actual lift

Is there a relationship between how deep the discount is and how much demand increases? This is the price elasticity question — central to Project 5.

In [ ]:
disc_lift = q(f"""
SELECT
    ROUND(discount_pct * 100 / 5) * 5 AS discount_bucket_pct,
    ROUND(AVG(lift_factor), 3) AS avg_lift,
    COUNT(*) AS n
FROM {SCHEMA}.v_promotion_lift
WHERE lift_factor IS NOT NULL AND lift_factor BETWEEN 0.5 AND 5.0
  AND discount_pct > 0
GROUP BY discount_bucket_pct
ORDER BY discount_bucket_pct
""")

fig, ax = plt.subplots()
ax.plot(disc_lift['discount_bucket_pct'], disc_lift['avg_lift'], marker='o', color='steelblue')
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Average Lift Factor by Discount Depth')
ax.set_xlabel('Discount Bucket (%)')
ax.set_ylabel('Average Lift Factor')
plt.tight_layout()
plt.show()

print('\nKey insight: Does deeper discount always = higher lift?')
print(disc_lift[['discount_bucket_pct', 'avg_lift', 'n']].to_string(index=False))

## 6. Spoilage from post-promotion overstock

For perishable products, a promotion followed by a demand dip can cause spoilage. The excess inventory ordered for the promotion sits on the shelf and expires.

This is where promotion planning intersects with waste reduction — a real KPI at grocery retailers.

In [ ]:
spoilage_by_reason = q(f"""
SELECT
    sp.spoilage_reason,
    COUNT(*) AS events,
    SUM(sp.units_spoiled) AS total_units,
    ROUND(SUM(sp.cost_of_spoilage), 2) AS total_cost
FROM {SCHEMA}.spoilage sp
GROUP BY sp.spoilage_reason
ORDER BY total_cost DESC
""")
print('Spoilage by reason:')
print(spoilage_by_reason.to_string(index=False))

---
## Summary: What We've Built

At the end of Project 1, you have:

| Deliverable | Status |
|---|---|
| 17-table star schema in PostgreSQL | ✓ |
| 2 years of realistic simulated data | ✓ |
| Causal demand model (seasonality, weather, promotions) | ✓ |
| Realistic supplier profiles (National/Regional/Local) | ✓ |
| Stockouts emerging naturally from inventory simulation | ✓ |
| 5 analytical views for fast queries | ✓ |
| Data integrity test suite | ✓ |
| 5 EDA notebooks | ✓ |

## What's Next: Project 2 — "Why Did We Run Out?"

Now that we have a functioning data warehouse with realistic stockout patterns, we can build the Stockout Investigation Engine.

The question: **For a given SKU × Store × Date, why was the product unavailable?**

Possible root causes we can now investigate with this data:
- Unexpected demand spike (compare sales to rolling average)
- Inaccurate forecast
- Supplier delay (check purchase_orders.is_late)
- DC shortage (check DC-level inventory_snapshots)
- Insufficient safety stock (compare units_on_hand to safety_stock)
- Transportation delay (check shipments.status)
- Promotion not planned for (check promotion_id on the stockout days)
- Weather event (check weather.severity)

Every one of these root causes is traceable through the tables we just built.